In [ ]:
# int_cols = ["order_number", "line_item", "customerkey", "storekey", "productkey", "quantity"]
# date_cols = ["order_date", "delivery_date"]
# string_cols = ["currency_code"]
#
#
# def str_trim(df, cols):
#     for col in df.columns:
#         if col in cols:
#             df = df.withColumn(col, F.trim(F.col(col)))
#
#     return df
#
# def to_integer(df, cols):
#
#     for col in df.columns:
#         if col in cols:
#             df = df.withColumn(col, F.col(col).cast("int"))
#
#     return df
#
#
# def to_date(df, cols, format="M/d/yyyy"):
#
#
#     for col in df.columns:
#         if col in cols:
#             before_count = df.filter(F.col(col).isNotNull()).count()
#             df = df.withColumn(col, F.try_to_date(F.col(col), format))
#             after_count = df.filter(F.col(col).isNotNull()).count()
#
#             if after_count < before_count:
#                 raise ValueError(f"{col}: {before_count - after_count} values failed to parse as {format}")
#
#     return df
#
# sales_df_raw = str_trim(sales_df_raw, string_cols)
# sales_df_raw = to_integer(sales_df_raw, int_cols)
# sales_df_raw = to_date(sales_df_raw, date_cols)
#
# sales_df_raw.printSchema()


In [ ]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions as F

In [ ]:
spark = DatabricksSession.builder.serverless().profile("azure").getOrCreate()

In [ ]:
customers_bronze = spark.table("electronics.bronze.customers")
sales_bronze  = spark.table("electronics.bronze.sales")
products_bronze = spark.table("electronics.bronze.products")
stores_bronze = spark.table("electronics.bronze.stores")
exchange_rates_bronze = spark.table("electronics.bronze.exchange_rates")

In [ ]:
customers_bronze.show(5, truncate=False)

In [ ]:
customers_bronze.printSchema()

In [ ]:
SILVER_TYPES = {
    "customers" : {
        "int": ["customerkey"],
        "date": ["birthday"],
        "string": ["gender", "name", "city", "state_code", "state", "zip_code", "country", "continent"]
    },
    "sales": {
        "int": ["order_number", "line_item", "customerkey", "storekey", "productkey", "quantity"],
        "date": ["order_date", "delivery_date"],
        "string": ["currency_code"]
    },
    "products": {
        "int": ["productkey", "subcategorykey", "categorykey"],
        "string": ["product_name", "brand", "color", "subcategory", "category"]
    },
    "stores": {
        "int": ["storekey", "square_meters"],
        "date": ["open_date"],
        "string": ["country", "state"]
    },
    "exchange_rates": {
        "date": ["date"],
        "string": ["currency"],
        "to_decimal": ["exchange"]
    }
}


In [ ]:
customers_bronze.filter(F.col('_rescued_data').isNotNull()).show(5, truncate=False)

In [ ]:
sales_bronze.show(5, truncate=False)

In [ ]:
sales_bronze.printSchema()

In [ ]:
# Checking there are values containing letters,so that I can turn them to int
sales_bronze.filter(F.expr("try_cast(order_number as int)").isNull()).show(truncate=False)

In [ ]:
products_bronze.show(5, truncate=False)

In [ ]:
products_bronze.printSchema()

In [ ]:
stores_bronze.show(5, truncate=False)

In [ ]:
stores_bronze.printSchema()

In [ ]:
exchange_rates_bronze.show(5, truncate=False)

In [ ]:
SILVER_TYPES = {
    "customers" : {
        "int": ["customerkey"],
        "date": ["birthday"],
        "string": ["gender", "name", "city", "state_code", "state", "zip_code", "country", "continent"]
    },
    "sales": {
        "int": ["order_number", "line_item", "customerkey", "storekey", "productkey", "quantity"],
        "date": ["order_date", "delivery_date"],
        "string": ["currency_code"]
    },
    "products": {
        "int": ["productkey", "subcategorykey", "categorykey"],
        "string": ["product_name", "brand", "color", "subcategory", "category"]
    },
    "stores": {
        "int": ["storekey", "square_meters"],
        "date": ["open_date"],
        "string": ["country", "state"]
    },
    "exchange_rates": {
        "date": ["date"],
        "string": ["currency"],
    }
}


def clean_money(col_name):
    return F.regexp_replace(F.col(col_name), r"[$,\s]", "").cast("decimal(10,2)")

def to_int(df, cols):
    for col in df.columns:
        if col in cols:
            df = df.withColumn(col, F.col(col).cast("int"))

    return df

def to_date(df, cols, fmt="M/d/yyyy"):

    for col in df.columns:
        if col in cols:
            before_count = df.filter(F.col(col).isNotNull()).count()
            df = df.withColumn(col, F.try_to_date(F.col(col), fmt))
            after_count = df.filter(F.col(col).isNotNull()).count()

            if after_count < before_count:
                raise ValueError(f"{col}: {before_count - after_count} values failed to parse as {fmt}")

    return df

def str_clean(df, cols):
    for col in df.columns:
        if col in cols:
            df = df.withColumn(col, F.trim(F.col(col)))

    return df

def apply_types(df, types_dict):
    d_type_mapper =  {
    "int": to_int,
    "date": to_date,
    "string": str_clean,
}
    for d_type, cols in types_dict.items():
        df = d_type_mapper[d_type](df, cols)

    return df


def type_sales(df):
    df = apply_types(df, SILVER_TYPES["sales"])

    return df

def type_customers(df):
    df = apply_types(df, SILVER_TYPES["customers"])

    return df.withColumn("age", F.floor(F.months_between(F.current_date(), "birthday") / 12))

def type_products(df):
    df = apply_types(df, SILVER_TYPES["products"])
    return (
        df
        .withColumn("unit_price_usd", clean_money("unit_price_usd"))
        .withColumn("unit_cost_usd", clean_money("unit_cost_usd"))
    )


def type_stores(df):
    df = apply_types(df, SILVER_TYPES["stores"])

    return df


def type_exchange_rates(df):
    df = apply_types(df, SILVER_TYPES["exchange_rates"])

    return (
        df
        .withColumn("exchange", F.col("exchange").cast("decimal(10,4)"))
    )

tables = ["sales", "customers", "products", "stores", "exchange_rates"]

clean_mapper = {
    "sales": type_sales,
    "customers": type_customers,
    "products": type_products,
    "stores": type_stores,
    "exchange_rates": type_exchange_rates,
}

cleaned_tables = {}

for table in tables:
    df = spark.table(f"electronics.bronze.{table}")
    df = clean_mapper[table](df)

    cleaned_tables[table] = df


for name, df in cleaned_tables.items():
    print(f"{name}:")
    df.printSchema()



In [ ]:
cleaned_tables["products"].show(5, truncate=False)

In [ ]:
cleaned_tables["exchange_rates"].show(5, truncate=False)

In [ ]:
cleaned_tables["stores"].show(5, truncate=False)

In [ ]:
cleaned_tables["sales"].show(5, truncate=False)

In [ ]:
cleaned_tables["customers"].show(5, truncate=False)

In [ ]:
cleaned_tables["products"].select("unit_price_usd", "unit_cost_usd").show(5, truncate=False)

In [ ]:
cleaned_tables["products"].filter(F.col("unit_price_usd").isNull()).count()

In [ ]:
cleaned_tables["products"].filter(F.col("unit_cost_usd").isNull()).count()

In [ ]:
cleaned_tables["exchange_rates"].filter(F.col("exchange").isNull()).count()

In [ ]:
from src.silver import build_silver

build_silver(spark, "electronics", "2026-09-18", "sales")

spark.table("electronics.silver.sales").printSchema()

In [ ]:
for entity in ["sales", "customers", "products", "stores", "exchange_rates"]:
    print(entity, build_silver(spark, "electronics", "2026-09-18", entity))

In [ ]:
cleaned_tables["customers"].show(5, truncate=False)

In [ ]:
sales_df = cleaned_tables["sales"]

date_range = (
    sales_df
    .agg(
        F.least(
            F.min(sales_df["order_date"]),
            F.min(sales_df["delivery_date"]),
        ).alias("min_date"),
        F.greatest(
            F.max(sales_df["order_date"]),
            F.max(sales_df["delivery_date"]),
        ).alias("max_date"),
    )
)

date_range.show()

In [ ]:
dim_date = (
    spark.sql("""
    SELECT explode(
            sequence(
            to_date('2015-01-01'),
            to_date('2026-12-31'),
            interval 1 day
            )
    ) AS full_date
    """)
    .select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_sk"),
        "full_date",
        F.dayofmonth("full_date").alias("day_of_month"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.weekofyear("full_date").alias("week_of_year"),
        F.month("full_date").alias("month_number"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.quarter("full_date").alias("quarter"),
        F.year("full_date").alias("year"),
        F.when(
            F.dayofweek("full_date").isin([1, 7]),
            True
        ).otherwise(False).alias("is_weekend"),
    )
)

dim_date.show(5, truncate=False)

In [ ]:
dim_date.filter(F.col("is_weekend")).select("full_date", "day_name").show(5)
dim_date.count()

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS electronics.gold")
dim_date.write.mode("overwrite").saveAsTable("electronics.gold.dim_date")
spark.table("electronics.gold.dim_date").count()

In [ ]:
dim_product = (
    spark.table("electronics.silver.products")
    .withColumn("product_sk", F.xxhash64(F.col("productkey").cast("string")))
    .drop("_rescued_data", "_file_path", "_file_size", "_last_modified_at", "_ingested_at", "_run_date", "_silver_run_date")
)


dim_product.count()

In [ ]:
dim_product.write.mode("overwrite").saveAsTable("electronics.gold.dim_product")

d = spark.table("electronics.gold.dim_product")

print(d.count())
print(d.select("product_sk").distinct().count())

In [ ]:
store_silver = spark.table("electronics.silver.stores")
store_silver.show(5, truncate=False)

In [ ]:
store_silver.filter(F.col("storekey") == 0).count()

In [ ]:
store_silver.filter(F.col("square_meters").isNull()).count()

In [ ]:
store_silver.select("storekey").distinct().count()

In [ ]:
dim_store= (
    spark.table("electronics.silver.stores")
    .withColumn("store_sk", F.xxhash64(F.col("storekey").cast("string")))
    .withColumn("store_type", F.when(F.col("storekey") == 0, "Online").otherwise("Physical"))
    .drop("_rescued_data", "_file_path", "_file_size", "_ingested_at", "_run_date", "_silver_run_date")
)

dim_store.show(5, truncate=False)

In [ ]:
dim_store.write.mode("overwrite").saveAsTable("electronics.gold.dim_store")

s = spark.table("electronics.gold.dim_store")

print("Rows: ", s.count())
print("Surrogate Key: ", s.select("store_sk").distinct().count())
print("Online Stores: ", s.filter(F.col("store_type") == "Online").count())

In [ ]:
customers_silver = spark.table("electronics.silver.customers")
customers_silver.show(5, truncate=False)

In [ ]:
print("Rows: ", customers_silver.count())

In [ ]:
customers_silver.printSchema()

In [ ]:
CATALOG = "electronics"
RUN_DATE = "2026-09-19"

SCD2_COLS = ["city", "state_code", "state", "zip_code", "country", "continent"]
LINEAGE_DROP = ["_rescued_data", "_file_path", "_ingested_at", "_run_date", "_silver_run_date"]

def scd2_hash(cols):
    return F.xxhash64(F.concat_ws("|", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in cols]))

# Initial Load

def init_dim_customer(spark, catalog):
    df = (
        spark.table(f"{catalog}.silver.customers")
        .drop(*LINEAGE_DROP)
        .withColumn("customer_sk", scd2_hash(["customerkey"] + SCD2_COLS))
        .withColumn("change_hash", scd2_hash(SCD2_COLS))
        .withColumn("valid_from", F.lit("1900-01-01").cast("date"))
        .withColumn("valid_to", F.lit("9999-12-31").cast("date"))
        .withColumn("is_current", F.lit(True))
    )

    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.gold.dim_customer")

    return spark.table(f"{catalog}.gold.dim_customer").count()

# Merge

def merge_dim_customer(spark, catalog, run_date):
    target_name = f"{catalog}.gold.dim_customer"

    source = (
        spark.table(f"{catalog}.silver.customers")
        .drop(*LINEAGE_DROP)
        .withColumn("change_hash", scd2_hash(SCD2_COLS))
    )

    current = spark.table(target_name).filter("is_current")

    changed = (
        source.alias("s")
        .join(current.alias("t"), "customerkey")
        .filter(F.col("s.change_hash") != F.col("t.change_hash"))
        .select("s.*")
    )

    new = source.join(current, "customerkey", "left_anti")

    staged = (
        changed.withColumn("merge_key", F.col("customerkey"))
        .unionByName(changed.withColumn("merge_key", F.lit(None).cast("int")))
        .unionByName(new.withColumn("merge_key", F.lit(None).cast("int")))
        .withColumn("customer_sk", scd2_hash(["customerkey"] + SCD2_COLS))
        .withColumn("valid_from", F.lit(run_date).cast("date"))
        .withColumn("valid_to", F.lit("9999-12-31").cast("date"))
        .withColumn("is_current", F.lit(True))
    )

    staged.createOrReplaceTempView("staged_customers")


    insert_cols = [c for c in staged.columns if c != "merge_key"]

    col_list = ", ".join(insert_cols)
    val_list = ", ".join(f"s.{c}" for c in insert_cols)

    spark.sql(f"""
    MERGE INTO {target_name} AS t
    USING staged_customers AS s
        ON t.customerkey = s.merge_key AND t.is_current
    WHEN MATCHED THEN UPDATE SET
        t.valid_to = date_sub(to_date('{run_date}'), 1),
        t.is_current = false
    WHEN NOT MATCHED THEN INSERT ({col_list}) VALUES ({val_list})
""")

    return spark.table(target_name).count()



In [ ]:
print("initial:", init_dim_customer(spark, CATALOG))

In [ ]:
spark.sql(f"UPDATE {CATALOG}.silver.customers SET city = 'Berlin' WHERE customerkey = 1269051")

spark.sql(f"SELECT customerkey, city, country FROM {CATALOG}.silver.customers WHERE customerkey = 1269051").show()

In [ ]:
print("after merge:", merge_dim_customer(spark, CATALOG, RUN_DATE))

spark.sql(f"""
    SELECT customerkey, city, valid_from, valid_to, is_current
    FROM {CATALOG}.gold.dim_customer
    WHERE customerkey = 1269051
    ORDER BY valid_from
""").show()

In [ ]:
print("second merge:", merge_dim_customer(spark, CATALOG, RUN_DATE))

In [ ]:
spark.sql(f"""
    SELECT COUNT(*) AS current_rows,
           COUNT(DISTINCT customerkey) AS distinct_customers
    FROM {CATALOG}.gold.dim_customer
    WHERE is_current
""").show()

In [ ]:
spark.sql(f"""
    SELECT COUNT(*) FROM {CATALOG}.gold.dim_customer
    GROUP BY customer_sk HAVING COUNT(*) > 1
""").show()

In [ ]:
sales_silver = spark.table(f"{CATALOG}.silver.sales")
sales_silver.show(5, truncate=False)

In [ ]:
exchange_rates_silver = spark.table(f"{CATALOG}.silver.exchange_rates")
products_silver = spark.table(f"{CATALOG}.silver.products")
dim_date = spark.table(f"{CATALOG}.gold.dim_date")

In [ ]:
# COLS_TO_DROP = ["order_date", "delivery_date", "customerkey", "storekey", "productkey", "currency_code", "gender", "name", "city", "state_code", "state", "zip_code", "country", "continent", "birthday", "change_hash", "valid_from", "valid_to", "is_current", "product_name", "brand", "color", "subcategory", "category", "category", "full_date", "day_of_month", "day_name", "week_of_year", "month_number", "month_name", "quarter", "year", "is_weekend", "square_meters", "open_date", "store_type"] + LINEAGE_DROP


dim_customer = spark.table(f"{CATALOG}.gold.dim_customer")

fct_sales = (
    sales_silver.alias("s")
 .join(dim_customer.alias("c"),
       (F.col("s.customerkey") == F.col("c.customerkey")) &
       (F.col("s.order_date") >= F.col("c.valid_from")) &
       (F.col("s.order_date") <= F.col("c.valid_to")),
       "left"
       )
    .join(dim_product.alias("p"),
          F.col("s.productkey") == F.col("p.productkey"),
          "left"
          )
    .join(dim_store.alias("st"),
          (F.col("s.storekey") == F.col("st.storekey")),
          "left"
          )
    .join(dim_date.alias("od"),
          (F.col("s.order_date") == F.col("od.full_date")),
          "left"
          )
    .join(dim_date.alias("dd"),
          (F.col("s.delivery_date") == F.col("dd.full_date")),
          "left"
          )
    .withColumn("revenue_usd", (F.col("s.quantity") * F.col("p.unit_price_usd")))
    .withColumn("cost_usd", (F.col("s.quantity") * F.col("p.unit_cost_usd")))
    .withColumn("margin_usd", (F.col("revenue_usd") - F.col("cost_usd")))
    .select(
        F.col("s.order_number").alias("order_number"),
        F.col("s.line_item").alias("line_item"),
        F.col("c.customer_sk").alias("customer_sk"),
        F.col("p.product_sk").alias("product_sk"),
        F.col("st.store_sk").alias("store_sk"),
        F.col("od.date_sk").alias("order_date_sk"),
        F.col("dd.date_sk").alias("delivery_date_sk"),
        F.col("s.quantity").alias("quantity"),
        F.col("p.unit_price_usd").alias("unit_price_usd"),
        F.col("p.unit_cost_usd").alias("unit_cost_usd"),
        F.col("revenue_usd"),
        F.col("cost_usd"),
        F.col("margin_usd"),
    )
)

In [ ]:
fct_sales.show(5, truncate=False)

In [ ]:
fct_sales.count()

In [ ]:
fct_sales.filter(F.col("delivery_date_sk").isNotNull()).count()

In [ ]:
for c in ["customer_sk", "product_sk", "store_sk", "order_date_sk"]:
    print(f"{c} Null Values: ", fct_sales.filter(F.col(c).isNull()).count())

In [ ]:
fct_sales.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.fct_sales")

In [ ]:
spark.sql(f"""
    SELECT d.year, p.category,
           SUM(f.revenue_usd) AS revenue,
           SUM(f.margin_usd)  AS margin
    FROM {CATALOG}.gold.fct_sales f
    JOIN {CATALOG}.gold.dim_date d    ON f.order_date_sk = d.date_sk
    JOIN {CATALOG}.gold.dim_product p ON f.product_sk = p.product_sk
    GROUP BY d.year, p.category
    ORDER BY d.year, revenue DESC
""").show(20)

In [ ]:
spark.sql(f"""
    SELECT c.city, COUNT(*) AS orders
    FROM {CATALOG}.gold.fct_sales f
    JOIN {CATALOG}.gold.dim_customer c ON f.customer_sk = c.customer_sk
    WHERE c.customerkey = 1269051
    GROUP BY c.city
""").show()

In [ ]:
def assert_unique_key(df, key_col, table_name=""):
    total = df.count()
    distinct = df.select(key_col).distinct().count()
    if total != distinct:
        raise ValueError(
            f"{table_name}: {total - distinct} duplicate values in {key_col}"
        )
    return df


def assert_one_current_version(df, business_key, table_name=""):
    current = df.filter("is_current")
    n_current = current.count()
    n_keys = current.select(business_key).distinct().count()
    if n_current != n_keys:
        raise ValueError(
            f"{table_name}: {n_current} current rows for {n_keys} distinct {business_key}"
        )
    return df


def assert_no_null_keys(df, key_cols, table_name=""):
    for c in key_cols:
        nulls = df.filter(F.col(c).isNull()).count()
        if nulls:
            raise ValueError(f"{table_name}: {nulls} null values in {c}")
    return df


def assert_row_count_matches(df, expected, table_name=""):
    actual = df.count()
    if actual != expected:
        raise ValueError(
            f"{table_name}: expected {expected} rows, got {actual}"
        )
    return df

In [ ]:
assert_unique_key(dim_product, "product_sk", "dim_product")
assert_unique_key(dim_store, "store_sk", "dim_store")
assert_unique_key(dim_date, "date_sk", "dim_date")
assert_unique_key(spark.table(f"{CATALOG}.gold.dim_customer"), "customer_sk", "dim_customer")

assert_one_current_version(spark.table(f"{CATALOG}.gold.dim_customer"), "customerkey", "dim_customer")

assert_no_null_keys(fct_sales, ["customer_sk", "product_sk", "store_sk", "order_date_sk"], "fct_sales")
assert_row_count_matches(fct_sales, spark.table(f"{CATALOG}.silver.sales").count(), "fct_sales")

In [ ]:
from src.gold import build_gold
build_gold(spark, "electronics", "2026-09-20")

In [ ]:
spark.stop()